# Chinese -> Tao: fine-tune NLLB (Google Colab)

Part 1 of 2. Run this notebook on a **Colab GPU runtime**. It fine-tunes NLLB on data **A**, saves the model to Google Drive and offers it as a download.
Part 2 (`tao_nllb_gemma.ipynb`) runs locally: it loads the model saved here and does the NLLB -> Gemma 4 post-editing evaluation.

**Expected data layout** (copy the repo's `data/` to `MyDrive/tulun/data/`):

```
data/finetune/   A  fine-tuning pairs (.json, .jsonl or .csv)
```

Records need `chinese_sentence` and `tao_sentence` (aliases `zh` / `tao` also work). Other keys are ignored.

**Things to know**
- Tao (Yami) is not one of NLLB-200's languages. The notebook adds a new language token `tao_Latn` and initializes its embedding from a related language (`INIT_FROM_LANG`).
- Keep the evaluation set B out of A. Part 2 warns about exact duplicates; for Bible-style data, also split by book/chapter.
- Output: `MyDrive/tulun/models/nllb-tao-ft/`. Sync it to the repo's `models/nllb-tao-ft/` (Google Drive for desktop, or the zip download at the end).

## 0. Setup

In [ ]:
%pip install -q transformers datasets accelerate sentencepiece

In [ ]:
import os, sys, json, csv, shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/tulun")

FINETUNE_DIR = ROOT / "data" / "finetune"   # A
MODEL_DIR    = ROOT / "models" / "nllb-tao-ft"

# --- languages (must match tao_nllb_gemma.ipynb) ---
SRC_LANG = "zho_Hant"        # use "zho_Hans" for simplified Chinese
TGT_LANG = "tao_Latn"        # NOT in NLLB-200: added as a new token during fine-tuning
INIT_FROM_LANG = "tgl_Latn"  # existing language whose embedding initializes tao_Latn (Austronesian)

# --- fine-tuning ---
BASE_MODEL = "facebook/nllb-200-distilled-600M"
MAX_LEN = 128
EPOCHS = 5
LEARNING_RATE = 1e-4
BATCH_SIZE = 8
GRAD_ACCUM = 4
VAL_FRACTION = 0.05
FORCE_RETRAIN = False

## 1. Load and check the data

In [ ]:
def read_records(path: Path):
    ext = path.suffix.lower()
    if ext == ".jsonl":
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    if ext == ".json":
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, list) else list(data.values())
    if ext == ".csv":
        with open(path, encoding="utf-8", newline="") as f:
            return list(csv.DictReader(f))
    return []

def _first(row, keys):
    for k in keys:
        if row.get(k):
            return str(row[k]).strip()
    return ""

def read_folder(folder: Path):
    if not folder.exists():
        raise FileNotFoundError(f"{folder} not found")
    rows = []
    for p in sorted(folder.iterdir()):
        rows += read_records(p)
    if not rows:
        raise ValueError(f"no .json/.jsonl/.csv records in {folder}")
    return rows

def load_pairs(folder: Path):
    """-> list of {'zh', 'tao'}; empty and duplicate pairs dropped."""
    out, seen = [], set()
    for r in read_folder(folder):
        zh = _first(r, ("chinese_sentence", "zh", "chinese"))
        tao = _first(r, ("tao_sentence", "tao", "yami"))
        if zh and tao and (zh, tao) not in seen:
            seen.add((zh, tao))
            out.append({"zh": zh, "tao": tao})
    return out

In [ ]:
data_a = load_pairs(FINETUNE_DIR)
print(f"A fine-tune: {len(data_a)}")
data_a[0]

## 2. Fine-tune NLLB on data A

Skipped automatically if `MODEL_DIR` already holds a model (set `FORCE_RETRAIN = True` to override). Raises if there is no GPU.

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)

def model_is_cached() -> bool:
    return (MODEL_DIR / "config.json").exists() and (MODEL_DIR / "tokenizer_config.json").exists()

def build_tokenizer_and_model():
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, src_lang=SRC_LANG)
    tok.add_tokens([TGT_LANG], special_tokens=True)          # new language code, no-op if it already exists
    tok.tgt_lang = TGT_LANG
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
    tgt_id = tok.convert_tokens_to_ids(TGT_LANG)
    if tgt_id >= model.get_input_embeddings().num_embeddings:
        model.resize_token_embeddings(len(tok))
    with torch.no_grad():                                    # warm-start the new token from a related language
        emb = model.get_input_embeddings().weight
        emb[tgt_id] = emb[tok.convert_tokens_to_ids(INIT_FROM_LANG)].clone()
    return tok, model

def train():
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU. In Colab: Runtime > Change runtime type > GPU.")
    tok, model = build_tokenizer_and_model()

    ds = Dataset.from_list(data_a).train_test_split(test_size=VAL_FRACTION, seed=42)
    ds = ds.map(lambda b: tok(b["zh"], text_target=b["tao"], max_length=MAX_LEN, truncation=True),
                batched=True, remove_columns=["zh", "tao"])
    print("label starts with:", tok.convert_ids_to_tokens(ds["train"][0]["labels"][:2]))   # expect tao_Latn

    bf16 = torch.cuda.is_bf16_supported()
    args = Seq2SeqTrainingArguments(
        output_dir="/content/nllb_ckpt",
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS, warmup_ratio=0.05, weight_decay=0.0, optim="adafactor",
        bf16=bf16, fp16=not bf16, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", logging_steps=25, report_to="none",
    )
    kwargs = dict(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
                  data_collator=DataCollatorForSeq2Seq(tok, model=model))
    try:
        trainer = Seq2SeqTrainer(processing_class=tok, **kwargs)     # transformers >= 4.46
    except TypeError:
        trainer = Seq2SeqTrainer(tokenizer=tok, **kwargs)
    trainer.train()

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(MODEL_DIR))
    tok.save_pretrained(str(MODEL_DIR))
    print("saved to", MODEL_DIR)

if model_is_cached() and not FORCE_RETRAIN:
    print("Fine-tuned model already cached at", MODEL_DIR, "- skipping training")
else:
    train()

### Download the model to your local machine

The model is already on Drive at `MyDrive/tulun/models/nllb-tao-ft/`. Either:
- let Google Drive for desktop sync `MyDrive/tulun/models/` to the repo's `models/` folder; or
- run the cell below (browser download of a ~2 GB zip), then unzip it to `models/nllb-tao-ft/` in the repo. Part 2 can also unpack `models/nllb-tao-ft.zip` for you.

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/nllb-tao-ft", "zip", MODEL_DIR)
files.download(archive)